# Gold de medidores — V-Energy

Lê a Silver de medidores (`v-energy.silver.tb_medidores`, PR #22) e produz:

1. `medidores_historico`: tabela do **contrato do portal** (`contracts/tabelas/medidores_historico.yaml`, `0.3.0-provisional`), com exatamente as colunas `id_medidor`, `id_uc`, `data_instalacao`, `data_retirada` e `tipo_medidor`, nessa ordem. Chave: `id_medidor` + `data_instalacao`.
2. `tb_medidores_historico`: tabela **analítica** (fora do contrato, prefixo `tb_`), com o medidor atual, a ordem do medidor na UC, a troca, as competências da instalação, da retirada e da troca, e a versão das regras. Atende a Visão 360, a fila de inspeção e o diagnóstico de receita.

A Gold não muda nenhum valor da Silver: só renomeia `tipo` para `tipo_medidor`, o nome do contrato, e `tipo_anterior` para `tipo_medidor_anterior`, e acrescenta colunas derivadas. As colunas, os tipos, a chave e as obrigatórias do contrato são lidos do YAML (`contracts/tabelas/medidores_historico.yaml`), e todo `id_uc` precisa existir na Silver de cadastro, de onde sai a `dim_uc`.

O padrão `modo=validar` não grava nada: **Run all** valida a Silver, monta e confere as duas tabelas em memória. Para gravar em `v-energy.gold`, o Job passa `modo=publicar` (ou troque o widget antes de executar), como nas Golds de GD e de continuidade. A exportação para o banco do portal é um passo separado (#96), que junta as tabelas do contrato de todas as Golds.

Regras e dicionário de dados: [`docs/gold/medidores.md`](../../docs/gold/medidores.md).

In [ ]:
"""Transformações e validações da Gold de medidores."""

import re
import uuid
from functools import reduce
from operator import or_

import yaml
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

VERSAO_REGRAS = 'GOLD_MEDIDORES_V1'
PADRAO_ID_MEDIDOR = r'^MD\d+$'
TIPOS_VALIDOS = ['ELETRONICO', 'ELETROMECANICO', 'SMART']
STATUS_VALIDOS = ['ATIVO', 'RETIRADO']
COLUNAS_SILVER = [
    'id_medidor',
    'id_uc',
    'modelo',
    'tipo',
    'status',
    'data_instalacao',
    'data_retirada',
    'id_medidor_anterior',
    'tipo_anterior',
]
# Tipos do YAML do contrato no Spark; numeric(p,s) vira decimal(p,s)
TIPOS_CONTRATO = {'text': 'string', 'date': 'date', 'integer': 'int', 'boolean': 'boolean'}
# Commits que não mudam dados; o auto compaction e a predictive optimization do Databricks
# podem gravá-los logo depois de uma escrita
OPERACOES_MANUTENCAO = {'OPTIMIZE', 'VACUUM START', 'VACUUM END'}
COMENTARIOS = {
    'id_medidor': 'Medidor (chave do contrato, com data_instalacao).',
    'id_uc': 'Unidade consumidora onde o medidor está ou esteve instalado.',
    'data_instalacao': 'Data de instalação na UC (chave do contrato, com id_medidor).',
    'data_retirada': 'Data de retirada por troca ou desativação; nula para o medidor atual.',
    'tipo_medidor': 'Tecnologia do medidor: ELETRONICO, ELETROMECANICO ou SMART.',
    'modelo': 'Modelo do medidor (ex.: EL-3000).',
    'status': 'ATIVO (instalado hoje) ou RETIRADO.',
    'flag_medidor_atual': 'True para o medidor instalado hoje na UC (exatamente 1 por UC).',
    'ordem_na_uc': 'Posição do medidor na UC: 1 = primeiro instalado, 2 = o que o substituiu...',
    'competencia_instalacao': 'Competência da instalação (primeiro dia do mês).',
    'competencia_retirada': 'Competência da retirada (primeiro dia do mês); nula no medidor atual.',
    'flag_instalado_em_troca': 'True se o medidor entrou substituindo outro da mesma UC.',
    'competencia_troca': (
        'Competência da troca: mês da retirada do medidor anterior, o mesmo das leituras com '
        'flag_troca_medidor. Nula se o medidor não entrou em troca.'
    ),
    'id_medidor_anterior': 'Medidor substituído por este; nulo se for o primeiro da UC.',
    'tipo_medidor_anterior': 'Tecnologia do medidor substituído; nula se for o primeiro da UC.',
    'versao_regras_gold': 'Versão das regras da Gold que gerou a linha.',
}
# Colunas da analítica depois das do contrato, na ordem do YAML
COLUNAS_SO_ANALITICAS = [
    'modelo',
    'status',
    'flag_medidor_atual',
    'ordem_na_uc',
    'competencia_instalacao',
    'competencia_retirada',
    'flag_instalado_em_troca',
    'competencia_troca',
    'id_medidor_anterior',
    'tipo_medidor_anterior',
    'versao_regras_gold',
]


def _exigir_colunas(df: DataFrame, colunas: list[str], nome: str) -> None:
    ausentes = set(colunas) - set(df.columns)
    if ausentes:
        raise ValueError(f'{nome}: colunas ausentes: {sorted(ausentes)}')


def _contar(df: DataFrame, regras: dict) -> tuple[dict, int]:
    """Conta, numa passada, as linhas e as violações de cada regra (verdadeira = violação)."""
    contagens = (
        df.agg(
            F.count('*').alias('_linhas'),
            *[F.count(F.when(condicao, 1)).alias(regra) for regra, condicao in regras.items()],
        )
        .first()
        .asDict()
    )
    return contagens, contagens.pop('_linhas')


def _falhar_se(contagens: dict, nome: str) -> dict:
    falhas = {regra: n for regra, n in contagens.items() if n}
    if falhas:
        raise ValueError(f'{nome} reprovada: {falhas}')
    return contagens


def _comentar(df: DataFrame) -> DataFrame:
    for coluna in df.columns:
        df = df.withMetadata(coluna, {'comment': COMENTARIOS[coluna]})
    return df


def ler_contrato(caminho) -> dict:
    """Colunas (nome e tipo Spark, na ordem), chave, obrigatórias e padrão do id_uc do YAML."""
    with open(caminho, encoding='utf-8') as arquivo:
        spec = yaml.safe_load(arquivo)
    colunas = []
    for nome, coluna in spec['columns'].items():
        # A regra R1 do contrato exige type e nullable em toda coluna
        if 'type' not in coluna or 'nullable' not in coluna:
            raise ValueError(f'{caminho}: coluna {nome} sem type ou nullable')
        decimal = re.fullmatch(r'numeric\((\d+),\s*(\d+)\)', coluna['type'])
        if decimal:
            colunas.append((nome, f'decimal({decimal[1]},{decimal[2]})'))
        elif coluna['type'] in TIPOS_CONTRATO:
            colunas.append((nome, TIPOS_CONTRATO[coluna['type']]))
        else:
            raise ValueError(f'{caminho}: tipo {coluna["type"]} de {nome} sem equivalente no Spark')
    return {
        'versao': spec['version'],
        'colunas': colunas,
        'chave': spec['primary_key'],
        'obrigatorias': [n for n, coluna in spec['columns'].items() if not coluna['nullable']],
        # O pattern é opcional no contrato, como no importador
        'padrao_id_uc': spec['columns'].get('id_uc', {}).get('pattern'),
    }


def validar_fonte(silver: DataFrame) -> tuple[dict, int]:
    """Premissas da Silver que a Gold usa (a Silver já as garante) e as linhas dela."""
    _exigir_colunas(silver, COLUNAS_SILVER, 'Silver de medidores')
    por_id = Window.partitionBy('id_medidor')
    por_uc = Window.partitionBy('id_uc')
    df = silver.withColumn('_n_id', F.count('*').over(por_id)).withColumn(
        '_ativos', F.count_if(F.col('status') == 'ATIVO').over(por_uc)
    )
    contagens, linhas = _contar(
        df,
        {
            'id_medidor nulo ou fora do padrão MD + dígitos': F.col('id_medidor').isNull()
            | ~F.col('id_medidor').rlike(PADRAO_ID_MEDIDOR),
            'id_medidor repetido': F.col('_n_id') > 1,
            'id_uc nulo': F.col('id_uc').isNull(),
            'tipo fora do domínio': F.col('tipo').isNull() | ~F.col('tipo').isin(TIPOS_VALIDOS),
            'status fora do domínio': F.col('status').isNull()
            | ~F.col('status').isin(STATUS_VALIDOS),
            'data_instalacao nula': F.col('data_instalacao').isNull(),
            'data_retirada anterior à instalação': F.col('data_retirada')
            < F.col('data_instalacao'),
            'status incoerente com data_retirada': (
                (F.col('status') == 'RETIRADO') & F.col('data_retirada').isNull()
            )
            | ((F.col('status') == 'ATIVO') & F.col('data_retirada').isNotNull()),
            'UC sem exatamente 1 medidor ATIVO': F.col('_ativos') != 1,
        },
    )
    return _falhar_se(contagens, 'Silver de medidores'), linhas


def construir_gold_medidores(silver: DataFrame, contrato: dict) -> dict:
    """Monta a tabela do contrato e a analítica a partir da Silver de medidores."""
    # Mesma ordem da Silver: no empate da instalação, o medidor com retirada vem antes do atual
    por_uc = Window.partitionBy('id_uc').orderBy(
        'data_instalacao', F.col('data_retirada').asc_nulls_last(), 'id_medidor'
    )
    analitica = silver.select(
        'id_medidor',
        'id_uc',
        'data_instalacao',
        'data_retirada',
        F.col('tipo').alias('tipo_medidor'),
        'modelo',
        'status',
        (F.col('status') == 'ATIVO').alias('flag_medidor_atual'),
        F.row_number().over(por_uc).alias('ordem_na_uc'),
        F.trunc('data_instalacao', 'month').alias('competencia_instalacao'),
        F.trunc('data_retirada', 'month').alias('competencia_retirada'),
        F.col('id_medidor_anterior').isNotNull().alias('flag_instalado_em_troca'),
        # A troca fica no mês da retirada do anterior, como nas leituras (flag_troca_medidor):
        # a Silver aceita lacuna até a instalação do novo, que pode cair no mês seguinte.
        F.when(
            F.col('id_medidor_anterior').isNotNull(),
            F.trunc(F.lag('data_retirada').over(por_uc), 'month'),
        ).alias('competencia_troca'),
        'id_medidor_anterior',
        F.col('tipo_anterior').alias('tipo_medidor_anterior'),
        F.lit(VERSAO_REGRAS).alias('versao_regras_gold'),
    )
    colunas_contrato = [c for c, _ in contrato['colunas']]
    faltando = [c for c in colunas_contrato if c not in analitica.columns]
    if faltando:
        raise ValueError(f'O YAML do contrato pede colunas que a Gold não gera: {faltando}')
    # Uma coluna analítica que entrar no contrato (#111) passa a sair na posição do YAML
    extras = [c for c in COLUNAS_SO_ANALITICAS if c not in colunas_contrato]
    analitica = analitica.select(*colunas_contrato, *extras)
    return {
        'medidores_historico': _comentar(analitica.select(*colunas_contrato)),
        'tb_medidores_historico': _comentar(analitica),
    }


def validar_gold(
    tabelas: dict,
    linhas_silver: int,
    cadastro: DataFrame,
    contrato: dict,
    rotulo: str,
) -> tuple[dict, dict]:
    """Contrato, reconciliação com a Silver e coerência da tabela analítica.

    Recebe as linhas da Silver contadas em validar_fonte. Devolve as violações por regra e as
    linhas de cada tabela.
    """
    tabela_contrato = tabelas['medidores_historico']
    analitica = tabelas['tb_medidores_historico']
    esquema = [(c.name, c.dataType.simpleString()) for c in tabela_contrato.schema]

    por_chave = Window.partitionBy(*contrato['chave'])
    chave = ', '.join(contrato['chave'])
    c, linhas_contrato = _contar(
        tabela_contrato.withColumn('_n', F.count('*').over(por_chave)),
        {
            f'contrato: chave ({chave}) repetida': F.col('_n') > 1,
            'contrato: coluna obrigatória nula': reduce(
                or_, [F.col(x).isNull() for x in contrato['obrigatorias']], F.lit(False)
            ),
            'contrato: id_uc fora do padrão do YAML': (
                ~F.col('id_uc').rlike(contrato['padrao_id_uc'])
                if contrato['padrao_id_uc']
                else F.lit(False)
            ),
            'contrato: tipo_medidor fora do domínio': ~F.col('tipo_medidor').isin(TIPOS_VALIDOS),
        },
    )
    c['contrato: colunas, tipos ou ordem diferentes do YAML'] = int(esquema != contrato['colunas'])
    c['contrato: linhas diferentes da Silver'] = abs(linhas_contrato - linhas_silver)
    # O YAML declara references: [dim_uc, id_uc]; a dim_uc sai da Silver de cadastro
    c['contrato: id_uc sem cadastro'] = (
        tabela_contrato.select('id_uc')
        .distinct()
        .join(cadastro.select('id_uc'), 'id_uc', 'left_anti')
        .count()
    )

    por_uc = Window.partitionBy('id_uc')
    ordem = Window.partitionBy('id_uc').orderBy('ordem_na_uc')
    a, linhas_analitica = _contar(
        analitica.withColumn('_atuais', F.count_if('flag_medidor_atual').over(por_uc))
        .withColumn('_ultima', F.max('ordem_na_uc').over(por_uc))
        .withColumn('_status_anterior', F.lag('status').over(ordem))
        .withColumn('_id_anterior', F.lag('id_medidor').over(ordem))
        .withColumn('_tipo_anterior', F.lag('tipo_medidor').over(ordem))
        .withColumn('_retirada_anterior', F.lag('competencia_retirada').over(ordem)),
        {
            'analítica: UC sem exatamente 1 medidor atual': F.col('_atuais') != 1,
            'analítica: medidor atual não é o último da UC': F.col('flag_medidor_atual')
            & (F.col('ordem_na_uc') != F.col('_ultima')),
            'analítica: troca incoerente com ordem_na_uc': F.col('flag_instalado_em_troca')
            != (F.col('ordem_na_uc') > 1),
            'analítica: medidor anterior diferente da ordem ou não RETIRADO': F.col(
                'flag_instalado_em_troca'
            )
            & (
                (F.col('_id_anterior') != F.col('id_medidor_anterior'))
                | (F.col('_status_anterior') != 'RETIRADO')
            ),
            'analítica: competência diferente do mês da data': (
                F.col('competencia_instalacao') != F.trunc('data_instalacao', 'month')
            )
            | ~F.col('competencia_retirada').eqNullSafe(F.trunc('data_retirada', 'month')),
            'analítica: tipo_medidor_anterior diferente do tipo do medidor anterior': ~F.col(
                'tipo_medidor_anterior'
            ).eqNullSafe(F.when(F.col('flag_instalado_em_troca'), F.col('_tipo_anterior'))),
            'analítica: competencia_troca diferente da retirada do medidor anterior': ~F.col(
                'competencia_troca'
            ).eqNullSafe(F.when(F.col('flag_instalado_em_troca'), F.col('_retirada_anterior'))),
        },
    )
    a['analítica: linhas diferentes da Silver'] = abs(linhas_analitica - linhas_silver)
    linhas = {'medidores_historico': linhas_contrato, 'tb_medidores_historico': linhas_analitica}
    return _falhar_se({**c, **a}, f'Gold de medidores ({rotulo})'), linhas


def _commits_depois(spark, destino: str, anterior) -> list:
    """Commits do destino depois da versão anterior (todos, se a tabela não existia)."""
    return (
        spark.sql(f'DESCRIBE HISTORY {destino}')
        .filter(F.col('version') > (-1 if anterior is None else anterior))
        .select('version', 'operation', 'userMetadata')
        .collect()
    )


def _recuperar(spark, destino: str, anterior, execucao: str) -> None:
    """Volta o destino à versão de antes desta execução, se ninguém mais o gravou nesse meio.

    Depois da versão anterior só pode haver commits desta execução (com o id dela em
    userMetadata) e manutenção que não muda dados. Outra escrita seria descartada por um
    RESTORE às cegas.
    """
    if not spark.catalog.tableExists(destino):
        return
    commits = _commits_depois(spark, destino, anterior)
    de_fora = [
        (c['version'], c['operation'])
        for c in commits
        if c['userMetadata'] != execucao and c['operation'] not in OPERACOES_MANUTENCAO
    ]
    if de_fora:
        raise RuntimeError(
            f'{destino}: outra escrita entrou junto com esta execução (versão anterior '
            f'{anterior}, commits de fora {de_fora}); recupere à mão'
        )
    if not any(c['userMetadata'] == execucao for c in commits):
        # A escrita desta execução não chegou a confirmar uma versão
        return
    if anterior is None:
        spark.sql(f'DROP TABLE IF EXISTS {destino}')
    else:
        spark.sql(f'RESTORE TABLE {destino} TO VERSION AS OF {anterior}')


def publicar_delta(spark, tabelas: dict, destinos: dict, validar) -> tuple:
    """Publica e confere as tabelas Gold; recupera os destinos afetados em caso de falha.

    Cada escrita leva o id da execução em userMetadata, que identifica no histórico Delta as
    versões gravadas por ela.
    """
    execucao = f'{VERSAO_REGRAS}:{uuid.uuid4()}'
    anteriores = {}
    for nome in tabelas:
        destino = destinos[nome]
        if spark.catalog.tableExists(destino):
            detalhe = spark.sql(f'DESCRIBE DETAIL {destino}').first()
            if detalhe['format'] != 'delta':
                raise ValueError(f'O destino precisa ser Delta: {destino}')
            anteriores[nome] = spark.sql(f'DESCRIBE HISTORY {destino} LIMIT 1').first()['version']
        else:
            anteriores[nome] = None

    afetadas = []
    persistidas, versoes = {}, {}
    try:
        for nome, df in tabelas.items():
            destino = destinos[nome]
            afetadas.append(nome)
            (
                df.write.format('delta')
                .mode('overwrite')
                .option('overwriteSchema', 'true')
                .option('userMetadata', execucao)
                .saveAsTable(destino)
            )
            # A versão desta escrita, e não a última: um OPTIMIZE automático pode vir depois
            proprias = [
                c['version']
                for c in _commits_depois(spark, destino, anteriores[nome])
                if c['userMetadata'] == execucao
            ]
            if len(proprias) != 1:
                raise ValueError(f'{destino}: esperava 1 versão desta execução, achei {proprias}')
            versao = proprias[0]
            persistido = spark.read.option('versionAsOf', versao).table(destino)
            esperado = [(c.name, c.dataType.simpleString()) for c in df.schema]
            gravado = [(c.name, c.dataType.simpleString()) for c in persistido.schema]
            if esperado != gravado:
                raise ValueError(f'Schema persistido divergente: {destino}')
            if (
                df.exceptAll(persistido).limit(1).count()
                or persistido.exceptAll(df).limit(1).count()
            ):
                raise ValueError(f'Conteúdo persistido divergente: {destino}')
            persistidas[nome] = persistido
            versoes[nome] = {'tabela': destino, 'versao_delta': versao}
        validar(persistidas)
    except Exception as erro:
        falhas = []
        for nome in reversed(afetadas):
            destino = destinos[nome]
            try:
                _recuperar(spark, destino, anteriores[nome], execucao)
            except Exception as falha:
                falhas.append(RuntimeError(f'Falha ao recuperar {destino}: {falha}'))
        if falhas:
            raise ExceptionGroup(
                'Publicação falhou e a recuperação ficou incompleta', [erro, *falhas]
            ) from erro
        raise
    return persistidas, versoes

In [ ]:
# ruff: noqa: F821
from pathlib import Path

parametros = {
    'tabela_medidores': 'v-energy.silver.tb_medidores',
    # De onde sai a dim_uc; quando a dim_uc for publicada, aponte para ela
    'tabela_cadastro': 'v-energy.silver.tb_unidades_consumidoras',
    # Relativo à pasta do notebook na pasta Git do repositório
    'contrato_yaml': '../../contracts/tabelas/medidores_historico.yaml',
    'schema_gold': 'v-energy.gold',
    'modo': 'validar',
}
for nome, padrao in parametros.items():
    dbutils.widgets.text(nome, padrao)
config = {nome: dbutils.widgets.get(nome).strip() for nome in parametros}
if config['modo'] not in {'validar', 'publicar'}:
    raise ValueError('modo deve ser validar ou publicar')


def qualificar(nome, partes=3):
    segmentos = nome.split('.')
    if len(segmentos) != partes or any(
        not re.fullmatch(r'[A-Za-z0-9_-]+', segmento) for segmento in segmentos
    ):
        raise ValueError(f'Identificador inválido: {nome}')
    return '.'.join(f'`{segmento}`' for segmento in segmentos)


schema_gold = qualificar(config['schema_gold'], 2)
# O Unity Catalog não diferencia maiúsculas: `Silver` é o schema silver
if config['schema_gold'].split('.')[1].lower() in {'silver', 'bronze', 'default'}:
    raise ValueError('O destino deve ser um schema próprio da Gold')
especificacao = ler_contrato(Path(config['contrato_yaml']))
fontes = {}
print(
    f'Schema Gold: {schema_gold} | Modo: {config["modo"]} | Regras: {VERSAO_REGRAS} | '
    f'Contrato: {especificacao["versao"]}'
)
if config['modo'] == 'publicar':
    print(f'ATENÇÃO: modo=publicar. As tabelas Gold serão gravadas em {schema_gold}')

## 1. Leitura consistente da Silver (`VERSION AS OF`)

Fixa a versão Delta atual das Silvers de medidores e de cadastro no início da execução, então a validação, a montagem e a escrita leem exatamente o mesmo snapshot. Se uma Silver estiver em outro catálogo ou schema no seu workspace, informe o nome no widget `tabela_medidores` ou `tabela_cadastro`.

In [ ]:
# ruff: noqa: F821
def ler_snapshot(chave):
    tabela = qualificar(config[chave])
    if not spark.catalog.tableExists(tabela):
        raise ValueError(
            f'Tabela Silver não encontrada: {tabela}. '
            f'Publique a Silver correspondente ou ajuste o widget {chave}.'
        )
    detalhe = spark.sql(f'DESCRIBE DETAIL {tabela}').first()
    if detalhe['format'] != 'delta':
        raise ValueError(f'A fonte precisa ser Delta: {tabela}')
    historico = spark.sql(f'DESCRIBE HISTORY {tabela} LIMIT 1').first()
    fontes[chave] = {
        'tabela': config[chave],
        'id_delta': detalhe['id'],
        'versao_delta': historico['version'],
        'timestamp_commit': str(historico['timestamp']),
    }
    print(f'{chave}: {tabela} (versão {historico["version"]})')
    return spark.read.option('versionAsOf', historico['version']).table(tabela)


df_medidores_silver = ler_snapshot('tabela_medidores')
df_cadastro_silver = ler_snapshot('tabela_cadastro')
regras_fonte, linhas_silver = validar_fonte(df_medidores_silver)
tabelas = construir_gold_medidores(df_medidores_silver, especificacao)
regras_gold, linhas_gold = validar_gold(
    tabelas, linhas_silver, df_cadastro_silver, especificacao, 'em memória'
)

display(
    spark.createDataFrame(
        [(r, n, 'OK') for r, n in {**regras_fonte, **regras_gold}.items()],
        'regra string, violacoes long, status string',
    )
)
for nome, linhas in linhas_gold.items():
    print(f'[OK] {nome}: {linhas:,} registros validados em memória')

## 2. Verificação das perguntas de negócio do case

Confere direto nas tabelas Gold montadas:
- **Visão 360 (Camila Barreto, Atendimento):** qual medidor está instalado na UC, de que tecnologia e desde quando, e qual foi o histórico de trocas?
- **Parque de medidores e campanha de troca (Henrique Duarte e Rafael Coutinho):** quantos medidores atuais há de cada tecnologia, e quantas trocas para smart houve por competência?

In [ ]:
# ruff: noqa: F821
contrato = tabelas['medidores_historico']
analitica = tabelas['tb_medidores_historico']

print('=== 1) Visão 360: medidores da UC100015 (tabela do contrato e analítica) ===')
display(contrato.filter(F.col('id_uc') == 'UC100015').orderBy('data_instalacao'))
display(
    analitica.filter(F.col('id_uc') == 'UC100015')
    .select(
        'ordem_na_uc',
        'id_medidor',
        'tipo_medidor',
        'modelo',
        'data_instalacao',
        'data_retirada',
        'flag_medidor_atual',
        'id_medidor_anterior',
    )
    .orderBy('ordem_na_uc')
)

print('=== 2) Parque atual por tecnologia ===')
display(
    analitica.filter('flag_medidor_atual')
    .groupBy('tipo_medidor')
    .agg(
        F.count('*').alias('medidores_atuais'),
        F.count_if('flag_instalado_em_troca').alias('instalados_em_troca'),
    )
    .orderBy(F.desc('medidores_atuais'))
)

print('=== 3) Campanha de troca por competência e tecnologia substituída ===')
display(
    analitica.filter('flag_instalado_em_troca')
    .groupBy(
        'competencia_troca',
        'tipo_medidor_anterior',
        F.col('tipo_medidor').alias('tipo_novo'),
    )
    .count()
    .orderBy('competencia_troca', 'tipo_medidor_anterior')
)

## 3. Publicação Delta com `RESTORE`

A publicação só ocorre quando `modo=publicar`. Antes da primeira escrita, registra a versão atual de cada tabela de destino. Cada escrita leva o id da execução no `userMetadata` do commit Delta. Depois de gravar, confere o schema e o conteúdo gravados e repete as regras do contrato e da analítica sobre as tabelas gravadas. Se algo falhar, restaura os destinos afetados com `RESTORE TABLE ... TO VERSION AS OF` (ou `DROP TABLE IF EXISTS` na primeira carga), desde que, depois da versão registrada, só haja commits desta execução e de manutenção (`OPTIMIZE`, `VACUUM`).

In [ ]:
# ruff: noqa: F821
if config['modo'] == 'validar':
    print('Validação concluída. Nenhum schema ou tabela foi gravado.')
else:
    if any(dbutils.widgets.get(nome).strip() != valor for nome, valor in config.items()):
        raise ValueError('Parâmetros mudaram durante a execução. Execute novamente desde o início.')
    destinos = {nome: f'{schema_gold}.`{nome}`' for nome in tabelas}
    for chave, fonte in fontes.items():
        atual = spark.sql(f'DESCRIBE DETAIL {qualificar(config[chave])}').first()
        if atual['id'] != fonte['id_delta']:
            raise ValueError(f'A identidade da tabela fonte mudou: {chave}')
    # Comparação sem maiúsculas, como o Unity Catalog resolve os nomes
    if {d.lower() for d in destinos.values()} & {qualificar(config[c]).lower() for c in fontes}:
        raise ValueError('Uma tabela de destino coincide com uma fonte')

    spark.sql(f'CREATE SCHEMA IF NOT EXISTS {schema_gold}')
    persistidas, versoes_gold = publicar_delta(
        spark,
        tabelas,
        destinos,
        lambda gravadas: validar_gold(
            gravadas, linhas_silver, df_cadastro_silver, especificacao, 'tabela gravada'
        ),
    )
    for nome, info in versoes_gold.items():
        print(f'Publicada: {info["tabela"]} (versão {info["versao_delta"]})')